In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
# In Colab the package AND data/ must be present, so we clone the repo (Colab opens
# only the single .ipynb, not the repo). In CI/local the package is already installed
# from the checkout under test, so we skip the clone and exercise that code.
import os

try:
    import pocketlm  # already installed (local/CI) -> use the code under test
except ImportError:
    import subprocess
    import sys

    if not os.path.isdir("pocketlm-repo"):
        subprocess.check_call(
            ["git", "clone", "--depth", "1",
             "https://github.com/ni5h4nt/pocketlm", "pocketlm-repo"])
    os.chdir("pocketlm-repo")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
    import pocketlm  # noqa: F811

import torch

# nbmake runs a notebook from its own directory; anchor the cwd at the repo root
# (derived from the installed package) so CORPUS_PATH resolves in CI, locally, and
# in Colab (where the except-branch already chdir'd into the clone).
os.chdir(os.path.dirname(os.path.dirname(os.path.abspath(pocketlm.__file__))))

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = "data/corpus.txt"   # repo-root-relative; valid after the chdir above
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l3.4 Multi-head attention

One attention is one relationship. Multi-head runs several in parallel on
slices of the channels, so different heads can track different patterns (syntax,
subject agreement, position), then concatenates them back.

In [ ]:
from pocketlm import scaled_dot_product_attention

torch.manual_seed(0)
tokens, n_head, hs = 5, 4, 6
channels = n_head * hs

def to_heads(t):
    return t.view(1, tokens, n_head, hs).transpose(1, 2)   # [1, n_head, T, hs]

q = to_heads(torch.randn(1, tokens, channels))
k = to_heads(torch.randn(1, tokens, channels))
v = to_heads(torch.randn(1, tokens, channels))
out, w = scaled_dot_product_attention(q, k, v, causal=True)
combined = out.transpose(1, 2).contiguous().view(1, tokens, channels)
print("per-head out:", tuple(out.shape), " recombined:", tuple(combined.shape))

The recombined tensor is back to `[batch, tokens, channels]`, ready for the
next layer. The heads were a parallel detour, invisible from outside the block.

In [ ]:
assert tuple(out.shape) == (1, n_head, tokens, hs)
assert tuple(w.shape) == (1, n_head, tokens, tokens)
assert tuple(combined.shape) == (1, tokens, channels)